# 综合数据集生成器（第 3 周练习解答）

## 练习目标（理念）

用 **Gradio** 做一个网页工具：选「业务域 + AI 模型 + 行数」，调用 Claude / GPT / Gemini 生成**尼日利亚语境**的合成数据，预览表格并导出 CSV。

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型 API | Anthropic / OpenAI / Google Generative AI 三套调用封装 |
| Prompt 工程 | `build_prompt` 要求只返回 JSON 数组 |
| Gradio UI | `gr.Blocks`：下拉框、滑块、表格预览、文件下载 |
| 合成数据 | 按 Fintech / Healthcare 等 domain schema 生成 |

## 怎么跑

1. 安装依赖单元格后，确保环境变量里有 `ANTHROPIC_API_KEY`、`OPENAI_API_KEY`、`GEMINI_API_KEY`
2. 从上到下运行：schemas → callers → Gradio UI
3. 在界面选 Domain / Model / Rows，点 Generate，下载 CSV


In [ ]:
# ========== 安装依赖：三家模型 SDK + Gradio + pandas ==========
# 用 pip 安装 anthropic（Claude）、openai（GPT）、google-generativeai（Gemini）、
# gradio（网页 UI）、pandas（表格与 CSV）
!pip install anthropic openai google-generativeai gradio pandas


In [ ]:
# ========== 导入 + 校验三家 API 密钥（Environment Variables）==========

# 标准库 os：从环境变量读取密钥
import os
# Gradio：快速搭 Web UI
import gradio as gr
# Anthropic 官方 SDK：调用 Claude
import anthropic
# OpenAI 官方 SDK：调用 GPT
import openai
# Google Generative AI SDK：调用 Gemini（别名 genai）
import google.generativeai as genai
# pandas：把 JSON 列表变成 DataFrame，并导出 CSV
import pandas as pd
# json：解析模型返回的 JSON 文本
import json
# traceback：异常时打印完整堆栈，方便排查
import traceback

# 把三家密钥读进字典；注意 Google 这边用环境变量名 GEMINI_API_KEY，
# 但字典键写成 GOOGLE_API_KEY，仅作「缺哪把钥匙」的展示名
required_keys = {
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "OPENAI_API_KEY":    os.getenv("OPENAI_API_KEY"),
    "GOOGLE_API_KEY":    os.getenv("GEMINI_API_KEY"),
}

# 列表推导：值为空的键名收集起来 → 缺密钥名单
missing = [k for k, v in required_keys.items() if not v]

# 有缺失就立刻报错，避免后面调 API 时才失败
if missing:
    raise EnvironmentError(f"Missing API keys: {missing}")

# 全部找到时给控制台一个确认信号
print("All API keys found.")
# 用环境变量 GEMINI_API_KEY 配置 Google Generative AI 全局客户端
genai.configure(api_key=os.environ["GEMINI_API_KEY"])


In [ ]:
# ========== 业务域 schemas：每个 domain 的说明 + 列名清单 ==========
# 「域模式」：告诉模型要生成哪类尼日利亚数据、字段有哪些（列名保持英文，供 JSON keys 使用）
schemas = {
    # 金融科技：交易与客户
    "Fintech": {
        "description": "Nigerian financial transactions and customer data",
        "columns": ["transaction_id", "customer_name", "bank", "amount_naira", "transaction_type", "status", "city", "timestamp", "is_fraud"]
    },
    # 医疗：病患与诊断
    "Healthcare": {
        "description": "Nigerian patient records and diagnoses",
        "columns": ["patient_id", "name", "age", "gender", "state", "diagnosis", "hospital", "admission_date", "discharge_date", "outcome"]
    },
    # 农业：产量与作物
    "Agriculture": {
        "description": "Nigerian farm yield and crop data",
        "columns": ["farmer_id", "name", "state", "crop_type", "farm_size_hectares", "yield_kg", "season", "rainfall_mm", "fertilizer_used", "revenue_naira"]
    },
    # 电商：订单与顾客
    "E-commerce": {
        "description": "Nigerian online shopping orders and customers",
        "columns": ["order_id", "customer_name", "product", "category", "price_naira", "quantity", "city", "delivery_status", "payment_method", "order_date"]
    },
    # 物流：包裹与路线
    "Logistics": {
        "description": "Nigerian package delivery and route data",
        "columns": ["delivery_id", "sender", "receiver", "origin_city", "destination_city", "weight_kg", "distance_km", "status", "delivery_days", "cost_naira"]
    },
    # 教育：学生成绩与学校
    "Education": {
        "description": "Nigerian student performance and school data",
        "columns": ["student_id", "name", "age", "gender", "state", "school_type", "subject", "score", "grade", "year"]
    }
}

# 打印已加载的 domain 数量，确认字典就绪
print(f"Loaded {len(schemas)} domain schemas.")


In [ ]:
# ========== 提示生成器：按 domain + 行数拼出给模型的 user prompt ==========
# 函数：根据业务域和行数，拼出「只要 JSON、不要解释」的英文 prompt（字符串内容保持原样，影响模型行为）
def build_prompt(domain, num_rows):
    # 从全局 schemas 取出该域的 description 与 columns
    schema = schemas[domain]
    # 把列名列表拼成逗号分隔字符串，方便写进 prompt
    columns = ", ".join(schema["columns"])

    # 返回多行 f-string：行数、上下文、列名、规则（含「只返回 JSON 数组」）
    return f"""Generate {num_rows} rows of realistic synthetic data for Nigerian {domain} sector.
Context: {schema["description"]}
Columns: {columns}

Rules:
- Use real Nigerian names, states, and context
- Return only valid JSON: a list of {num_rows} objects
- Each object must have exactly these keys: {columns}
- No explanation, no markdown, just the JSON array
"""


In [ ]:
# ========== 模型调用者：Claude / GPT / Gemini 三套薄封装 ==========
# 「模特来电者」→ 各厂商 API 的调用函数；返回值都是模型原始文本

# Claude：用 Anthropic SDK 发一条 user 消息
def call_claude(prompt):
    # 创建 Anthropic 客户端（默认读 ANTHROPIC_API_KEY）
    client = anthropic.Anthropic()
    # messages.create：指定模型 id、最大 token、messages 列表
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=16000,
        messages=[{"role": "user", "content": prompt}]
    )
    # 取第一条 content 块的文本（假设是 text 块）
    return response.content[0].text


# GPT：用 OpenAI Chat Completions
def call_gpt(prompt):
    # 创建 OpenAI 客户端（默认读 OPENAI_API_KEY）
    client = openai.OpenAI()
    # chat.completions.create：模型 gpt-4o-mini，单条 user 消息
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=16000
    )
    # 取第一条 choice 的 message.content
    return response.choices[0].message.content


# Gemini：用 GenerativeModel.generate_content
def call_gemini(prompt):
    # 绑定具体模型 id
    model = genai.GenerativeModel("gemini-2.5-flash")
    # 一次生成，prompt 直接作为内容
    response = model.generate_content(prompt)
    # 返回响应文本
    return response.text


# UI 下拉显示名 → 对应调用函数；键名供 Gradio Dropdown 使用
MODEL_CALLERS = {
    "Claude":  call_claude,
    "GPT-4":   call_gpt,
    "Gemini":  call_gemini
}


In [ ]:
# ========== 数据生成器：prompt → 调模型 → 剥 markdown → DataFrame ==========

# 按 domain / 模型名 / 行数走完「生成 → 解析 → 表格」流水线
def generate_data(domain, model_name, num_rows):
    # 用 schemas 拼出给 LLM 的 prompt
    prompt = build_prompt(domain, num_rows)
    # 按 UI 选中的模型名取出调用函数
    caller = MODEL_CALLERS[model_name]

    # 调用对应厂商 API，得到原始字符串
    raw = caller(prompt)

    # 若模型包了 ```json ... ```，剥掉围栏再解析（removeprefix / removesuffix）
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    # 把 JSON 数组解析成 Python list[dict]
    data = json.loads(raw)
    # 转成 pandas DataFrame，便于预览与导出
    df = pd.DataFrame(data)

    # 返回表格对象
    return df


In [ ]:
# ========== 导出为 CSV：按 domain_模型名 命名文件 ==========

# 把 DataFrame 写到本地 csv，返回文件名字符串（供 Gradio File 组件下载）
def export_csv(df, domain, model_name):
    # 拼文件名；去掉连字符与空格，避免路径尴尬
    filename = f"{domain}_{model_name}_synthetic.csv".replace("-", "").replace(" ", "_")
    # index=False：不把行号写进 CSV
    df.to_csv(filename, index=False)

    # 把路径/文件名交给调用方
    return filename

# 测试：直接调用导出（依赖前面单元格已有名为 df 的 DataFrame）
export_csv(df, "Fintech", "Claude")


In [ ]:
# ========== Gradio UI：选择域/模型/行数 → 预览表 + 下载 CSV ==========
# Cell 9 — Gradio UI（兼容 Gradio 6.x）


# 按钮回调：组装 prompt、调模型、解析 JSON、导出 CSV；失败时转成 gr.Error
def run_generator(domain, model_name, num_rows):

    try:

        # 注意：此处第二个实参按原代码传入 model_name（与 cell 4 的 num_rows 形参名不同，逻辑保持原样）
        prompt = build_prompt(domain, model_name)


        # 按模型名取调用函数并执行
        caller = MODEL_CALLERS[model_name]
        raw = caller(prompt)



        # 剥掉可能的 markdown 代码围栏后 json.loads
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        data = json.loads(cleaned)


        # list[dict] → DataFrame 供界面预览
        df = pd.DataFrame(data)


        # 写 CSV，拿到文件路径给下载组件
        filepath = export_csv(df, domain, model_name)


        # 返回：(表格预览, 下载文件路径)
        return df, filepath

    # JSON 解析失败：打印原始内容并抛 Gradio 错误气泡
    except json.JSONDecodeError as e:
        print(f"[ERROR] JSON parse failed: {e}")
        print(f"[ERROR] Raw content was: {raw}")
        raise gr.Error(f"JSON parse error: {e}")
    # 其它异常：打印堆栈，再包装成 gr.Error
    except Exception as e:
        print(f"[ERROR] {traceback.format_exc()}")
        raise gr.Error(str(e))

# 用 Blocks 搭页面；title 出现在浏览器标签
with gr.Blocks(title="Nigeria Synthetic Data Generator") as app:
    # 页面内大标题（Markdown）
    gr.Markdown("## Nigeria Synthetic Data Generator")

    # 一行三控件：域、模型、行数
    with gr.Row():
        # Domain 下拉：选项来自 schemas 的键；默认 Fintech
        domain    = gr.Dropdown(choices=list(schemas.keys()), label="Domain", value="Fintech")
        # Model 下拉：选项来自 MODEL_CALLERS 的键；默认 Claude
        model     = gr.Dropdown(choices=list(MODEL_CALLERS.keys()), label="Model", value="Claude")
        # 行数滑块：5~500，步长 5，默认 10
        num_rows  = gr.Slider(minimum=5, maximum=500, step=5, value=10, label="Rows")

    # 触发按钮
    generate_btn = gr.Button("Generate")
    # 表格预览组件
    table        = gr.Dataframe(label="Preview")
    # 文件下载组件
    download     = gr.File(label="Download CSV")

    # 绑定点击：inputs 三控件 → outputs 表 + 文件
    generate_btn.click(
        fn=run_generator,
        inputs=[domain, model, num_rows],
        outputs=[table, download]
    )

# 启动本地 Gradio 服务（默认会打印访问 URL）
app.launch()
